# Ascent-DTwin — Anomaly Detection

Machine-learning anomaly detection on live twin telemetry pulled from the Ascent API.
The API already flags anomalies in real time (z-score, see the `anomaly` field per point and the twin health score).
Here we go one step further with an **Isolation Forest** trained on the recent history.

In [ ]:
import urllib.request, json

API = "http://ascent-api:8000"
try:
    print(urllib.request.urlopen(API + "/api/health", timeout=3).read().decode()[:200])
except Exception as e:
    print("ascent-api not reachable, falling back to localhost:", e)
    API = "http://localhost:8000"
    print(urllib.request.urlopen(API + "/api/health", timeout=3).read().decode()[:200])

In [ ]:
import pandas as pd

pts = json.loads(urllib.request.urlopen(API + "/api/twins/esp32-demo/telemetry?limit=500", timeout=10).read())
df = pd.DataFrame(pts)
df["time"] = pd.to_datetime(df["time"])
print(f"{len(df)} points, columns: {df.columns.tolist()}")
df.tail(3)

In [ ]:
from sklearn.ensemble import IsolationForest

feats = [c for c in ["temperature", "humidity", "pressure", "co2"] if c in df.columns]
X = df[feats].fillna(df[feats].mean())

model = IsolationForest(contamination=0.05, random_state=42)
df["anomaly_ml"] = model.fit_predict(X) == -1

print(f"ML anomalies: {df['anomaly_ml'].sum()} / {len(df)}")
print(f"API z-score anomalies: {df.get('anomaly', pd.Series(False)).sum()} / {len(df)}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["time"], df["temperature"], color="#0969da", lw=1)
an = df[df["anomaly_ml"]]
ax.scatter(an["time"], an["temperature"], color="#cf222e", s=40, label="ML anomaly")
ax.set_title("esp32-demo temperature — Isolation Forest anomalies (red)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Health score

The API computes a 0-100 health score per twin (freshness 40 + data volume 30 + anomaly rate 30):

In [ ]:
import json
h = json.loads(urllib.request.urlopen(API + "/api/twins/esp32-demo/health", timeout=5).read())
print(json.dumps(h, indent=2))